In [1]:
!pip install numpy matplotlib pandas jupyter torch torchvision tensorflow onnx onnxruntime onnx2tf

INFO: pip is looking at multiple versions of onnx2tf to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of onnx2tf to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ---------------------------------------- 0.0/21.0 MB ? eta -:--:--
     ---------------------------------------- 0.0/21.0 MB ? eta -:--:--
     ---------------------------------------- 0.0/21.0 MB ? eta -:--:--
     ---------------------------------------- 0.0/21.0 MB ? eta -:--:--
     ---------------------------------------- 0.0/21.0 MB ? eta -:--:--
     ---------------------------------------- 0.0/21.0 MB ? eta -:--:--
     ---------------------------

  error: subprocess-exited-with-error
  
  Getting requirements to build wheel did not run successfully.
  exit code: 1
  
  [23 lines of output]
  <string>:28: DeprecationWarning: Use shutil.which instead of find_executable
  fatal: not a git repository (or any of the parent directories): .git
  fatal: not a git repository (or any of the parent directories): .git
  Traceback (most recent call last):
    File "D:\Conda\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
      main()
    File "D:\Conda\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
      json_out["return_val"] = hook(**hook_input["kwargs"])
                               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    File "D:\Conda\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 143, in get_requires_for_build_wheel
      return hook(config_settings)
             ^^^^^^^^^^^^^^^^^^^^^
    File "C:\Users\hp\AppData

In [2]:
import numpy as np

In [18]:
tensor = np.array([1.0, -0.5, 3.2, -2.8, 0.0], dtype=np.float32)

scale = 0.025
zero_point = 0

In [20]:
def quantize_tensor(tensor, scale, zero_point):
    """q = round(x / scale) + zero_point"""

    scaled_tensor = tensor / scale

    rounded_tensor = np.round(scaled_tensor)

    quantized_tensor = rounded_tensor + zero_point

    quantized_tensor = np.clip(quantized_tensor, -128, 127)

    return quantized_tensor.astype(np.int8)

In [22]:
def dequantize_tensor(quantized_tensor, scale, zero_point):
    """x = (q - zero_point) * scale"""

    dequantized_tensor = (quantized_tensor.astype(np.float32) - zero_point) * scale

    return dequantized_tensor

In [24]:
quantized_tensor = quantize_tensor(tensor, scale, zero_point)

dequantized_tensor = dequantize_tensor(quantized_tensor, scale, zero_point)

In [26]:
absolute_error = np.abs(tensor - dequantized_tensor)
mae = np.mean(absolute_error)

max_error = np.max(absolute_error)

In [28]:
print("Original tensor:")
print(tensor)

print("\nQuantized tensor (INT8):")
print(quantized_tensor)

print("\nDequantized tensor (float):")
print(dequantized_tensor)

print("\nAbsolute error per element:")
print(absolute_error)

print("\nMean Absolute Error (MAE):")
print(mae)

print("\nMaximum Absolute Error:")
print(max_error)

Original tensor:
[ 1.  -0.5  3.2 -2.8  0. ]

Quantized tensor (INT8):
[  40  -20  127 -112    0]

Dequantized tensor (float):
[ 1.    -0.5    3.175 -2.8    0.   ]

Absolute error per element:
[0.        0.        0.0250001 0.        0.       ]

Mean Absolute Error (MAE):
0.005000019

Maximum Absolute Error:
0.025000095


## Observations

- The tensor was quantized using affine INT8 quantization with a scale of **0.025** and a zero point of **0**.
- Each floating-point value was divided by the scale, rounded to the nearest integer, and clipped to the INT8 range **[-128, 127]**.
- The value **3.2** exceeded the representable range after scaling (3.2 / 0.025 = 128), so it was clipped to **127**.
- During dequantization, the clipped value became **3.175**, resulting in an absolute error of approximately **0.025**.
- The remaining values were represented exactly because they mapped to valid INT8 values without clipping.
- The Mean Absolute Error (MAE) is very small, showing that quantization preserved most values accurately except for the clipped value.